In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fractions import Fraction
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, ScalarFormatter

def plot_ratio_line_subplots_beautiful(
    df: pd.DataFrame,
    remove_p = (0.6,),
    use_log_y: bool = True,
    bar_width: float = 0.06,
    jitter: float = 0.009,
    figsize_per_p: float = 6.5,
    annotate_best: bool = True,
    suptitle: str = "Scheme Comparison by Ratio",
    output_path: str = None
):
    def to_fraction_str(numer, denom):
        try:
            f = Fraction(int(numer), int(denom))
            return f"{f.numerator}/{f.denominator}"
        except Exception:
            return f"{numer}/{denom}"

    def parse_row(name):
        s = str(name).strip()

        m_agg = re.search(r'agg[_\- ]?mac[_\- ]?(\d+)', s, flags=re.IGNORECASE)
        if m_agg:
            n = int(m_agg.group(1))
            return "AGG_MAC", f"1/{n}", 1.0/n

        m_frac = re.search(r'(\d+)\s*/\s*(\d+)', s)
        if m_frac:
            a, b = int(m_frac.group(1)), int(m_frac.group(2))
            head = s[:m_frac.start()]
            head = re.sub(r'[_\-\s]+$', '', head)
            m_head = re.search(r'([A-Za-z0-9\-\(\)_]+)$', head)
            scheme = m_head.group(1) if m_head else head
            return scheme, to_fraction_str(a, b), (a / b if b else np.nan)

        if re.search(r'\bwhip', s, flags=re.IGNORECASE):
            return s, "1/1", 1.0
        if re.search(r'\btrad(itional)?\b', s, flags=re.IGNORECASE):
            return "Traditional", "1/1", 1.0

        if "golomb" in s.lower():
            return "Golomb-SP_MAC", "1/1", 1.0
        
        if s.lower() == "g-sidon_2":
            return "Sidon-SP_MAC", "1/1", 1.0

        scheme = re.split(r'[_\-]', s)[0] if s else s
        return scheme, "1/1", 1.0

    need = {"name","res","p"}
    if not need.issubset(df.columns):
        missing = need - set(df.columns)
        raise ValueError(f"df must contain columns {need}, missing: {missing}")

    data = df.copy()
    parsed = data["name"].apply(parse_row)
    data[["scheme","ratio_str","ratio_num"]] = pd.DataFrame(parsed.tolist(), index=data.index)

    if remove_p:
        data = data[~data["p"].isin(remove_p)]
    if data.empty:
        raise ValueError("No data left after filtering 'p'.")

    agg = (
        data.groupby(["p","ratio_num","ratio_str","scheme"], dropna=False)["res"]
            .mean()
            .reset_index()
    )

    p_values = sorted(agg["p"].unique(), reverse=1)
    schemes = sorted(agg["scheme"].unique(), key=lambda s: (not str(s).startswith("Whip"), str(s)))

    from matplotlib.colors import LinearSegmentedColormap
    import matplotlib.colors as mcolors

    # OptiMAC — vivid blue gradient, fully opaque & prominent
    opti_cmap = LinearSegmentedColormap.from_list('opti_blue', ['#64B5F6', '#1565C0', '#0D47A1'])
    all_ratios = agg[agg['scheme'] == 'OptiMAC']['ratio_num']
    if len(all_ratios) > 0:
        norm_ratio = mcolors.PowerNorm(gamma=0.4, vmin=all_ratios.min(), vmax=all_ratios.max())
    else:
        norm_ratio = mcolors.Normalize(0, 1)

    # SOTA — distinct warm/cool pastels, clearly different from each other
    sota_colors = {
        "AGG_MAC":       "#E8A87C",   # warm peach
        "Traditional":   "#95A5A6",   # cool grey
        "Sidon-SP_MAC":  "#C39BD3",   # soft lavender
        "Golomb-SP_MAC": "#F0B27A",   # warm sand
        "Whip_2":        "#76D7C4",   # mint
        "Whip_3":        "#F1948A",   # salmon pink
        "Whip_4":        "#7DCEA0",   # soft green
        "Whip_5":        "#F7DC6F",   # mellow yellow
    }
    sota_alpha = 0.55
    opti_alpha = 1.0

    hatch_for = {"Whip_2":"//", "Whip_3":"xx", "Whip_4":"..", "Whip_5":"++"}

    plt.rcParams.update({
        "font.family": "serif",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": "#555555", "axes.linewidth": 1.0,
        "axes.titlesize": 14, "axes.labelsize": 14,
        "xtick.labelsize": 13, "ytick.labelsize": 13,
        "legend.fontsize": 12,
        "axes.facecolor": "#FAFAFA",
        "figure.facecolor": "white",
    })

    ncols = len(p_values)
    # IEEE figure* style: wide and compact (spans two columns ~7.16in, here scaled up for clarity)
    fig_w = figsize_per_p * ncols
    fig_h = 3.8
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h), sharey=True, dpi=200)
    if ncols == 1:
        axes = [axes]

    for ax in axes:
        ax.grid(axis="y", alpha=0.3, linestyle="-", linewidth=0.4, color="#CCCCCC")
        ax.grid(axis="x", visible=False)
        ax.tick_params(axis='x', which='both', length=0)

    for col_idx, (ax, pval) in enumerate(zip(axes, p_values)):
        sub = agg[agg["p"] == pval].copy()
        ratios_sorted = sorted(np.round(np.linspace(0, 1.05, 4)))
        ax.set_xlim(-0.00, 1.11)

        n_series = len(schemes)
        offsets = {s: (i - (n_series - 1)/2) * jitter for i, s in enumerate(schemes)}

        drawn = {}
        for scheme in schemes:
            ssub = sub[sub["scheme"] == scheme]
            if ssub.empty:
                continue
            xs = ssub["ratio_num"].values + np.array([offsets[scheme]] * len(ssub))
            ys = ssub["res"].values

            is_opti = scheme == "OptiMAC"

            bars = ax.bar(
                xs, ys,
                width=bar_width,
                align="center",
                color=[opti_cmap(0.15 + 0.85 * norm_ratio(r)) for r in ssub["ratio_num"].values] if is_opti else sota_colors.get(scheme, '#BDBDBD'),
                edgecolor='#0D47A1' if is_opti else '#FFFFFF',
                linewidth=2.0 if is_opti else 0.8,
                alpha=opti_alpha if is_opti else sota_alpha,
                zorder=5 if is_opti else 2,
            )
            if str(scheme).startswith("Whip"):
                for b in bars:
                    b.set_hatch(hatch_for.get(scheme, ""))
                    b.set_edgecolor(sota_colors.get(scheme, '#BDBDBD'))

            for x, y, b in zip(xs, ys, bars):
                drawn.setdefault(round(float(x), 6), []).append((b, y, scheme))

        if annotate_best:
            for xkey, items in drawn.items():
                b, y, scheme = max(items, key=lambda t: t[1])
                b.set_linewidth(2.0)
                b.set_edgecolor("#222222")

        # Subplot label: (a), (b), (c)
        subplot_label = chr(ord('a') + col_idx)
        ax.set_title(f"({subplot_label}) attack rate = {np.round(1-pval,1)}", fontweight='bold', pad=10, fontsize=14)
        if use_log_y:
            ax.set_yscale("log")
            ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
            ax.yaxis.set_minor_locator(MaxNLocator(nbins=12))
            ax.yaxis.set_major_formatter(ScalarFormatter())
            ax.tick_params(axis='y', which='minor', length=0)
        else:
            ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

        ax.set_xticks(ratios_sorted)
        ax.set_xlabel("TMR", fontweight='bold', fontsize=14)

    axes[0].set_ylabel("Goodput (%)", fontweight='bold', fontsize=14)

    # Legend — single row above all subplots
    legend_handles, seen = [], set()
    for scheme in schemes:
        if scheme == "OptiMAC":
            patch = Patch(facecolor=opti_cmap(0.55), edgecolor='#0D47A1', linewidth=2.0, label='OptiMAC')
        else:
            patch = Patch(
                facecolor=sota_colors.get(scheme, '#BDBDBD'),
                edgecolor=sota_colors.get(scheme, '#BDBDBD') if scheme.startswith("Whip") else 'white',
                alpha=sota_alpha,
                hatch=hatch_for.get(scheme, ""),
                linewidth=0.8,
                label=scheme
            )
        if scheme not in seen:
            legend_handles.append(patch); seen.add(scheme)

    fig.legend(
        handles=legend_handles,
        loc="upper center",
        ncol=len(legend_handles),
        frameon=True,
        fancybox=True,
        shadow=False,
        edgecolor='#DDDDDD',
        facecolor='white',
        fontsize=11,
        bbox_to_anchor=(0.5, 1.10),
        handlelength=1.5,
        handleheight=1.0,
        columnspacing=1.0,
    )
    if suptitle:
        fig.suptitle(suptitle, y=1.14, fontsize=15, fontweight="bold")

    plt.tight_layout(rect=[0, 0, 1, 0.97], w_pad=2.5)

    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


In [ ]:
plot_ratio_line_subplots_beautiful(
    pd.read_csv(f'periodic_simulation_res_mSize_{32}_secReq_256.csv'),
    remove_p=(0.6,),       # removes attack rate 0.4
    use_log_y=False,
    bar_width=0.024,
    jitter=0.023,
    figsize_per_p=7.5,     # wider per subplot for IEEE figure* spanning two columns
    suptitle=""
)
